In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import re
import glob
import cv2


# Inputs
Set `LP_DIR` (the Lightning Pose run's `video_preds` folder) and recording
parameters. Everything else is auto-discovered. `DLC_LEGACY_FOLDER` is only
used by the standalone shuffle6/7/8 DLC-vs-DLC comparison cells further down
— it has nothing to do with the Lightning Pose correction pipeline itself.

In [ ]:
LP_DIR = r"\\wsl.localhost\Ubuntu\home\almog\LightPose\lightning-pose\scripts\configs\outputs\2026-08-29\18-17-58\video_preds"

VIDEO_SEARCH_DIRS = [
    r"C:\Users\Almog\Desktop\Neuropixel_2Cams_Gamma-AnA_July-2025-07-22\videos",
    r"C:\Maze\Video Data\DLC done",
    r"C:\Maze\Video Data\NPC4",
]

OUTPUT_DIR = os.path.join(LP_DIR, "corrected")   # DLC-format .h5/.csv land here, for NaviGraph

DLC_LEGACY_FOLDER = r"C:\\Maze\\Video Data\\DLC Done\\NPC4_new\\"   # only used by the shuffle6/7/8 DLC comparison cells below

FPS            = 40     # frames per second
CHUNK_SEC      = 10     # chunk duration used for interpolation context
LIKELIHOOD_THR = 0.85   # frames below this threshold are corrected
MIN_MOVEMENT_PX = 10    # chunks with less x/y range than this (in both axes) are left uncorrected
N_PLOT_CHUNKS  = 20     # most-affected chunks to plot per session


# Discover Sessions
Every `*.reencoded.csv` in `LP_DIR` (excluding the `_temporal_norm` variants,
which are a smoothed/normalized copy, not the raw model output) defines a
session. Each is paired with its video, when one exists, for the
maze-background plot — sessions without a matching video are still processed,
just without a maze background image.

In [ ]:
# Match Lightning Pose prediction CSVs, exclude the temporal-norm variants
lp_csvs = sorted(
    p for p in glob.glob(os.path.join(LP_DIR, "*.reencoded.csv"))
    if not p.endswith("_temporal_norm.csv")
)

sessions = []
for csv_path in lp_csvs:
    fname      = os.path.basename(csv_path)
    session_id = fname[: -len(".reencoded.csv")]

    video_path = None
    for folder in VIDEO_SEARCH_DIRS:
        candidate = os.path.join(folder, session_id + ".mp4")
        if os.path.exists(candidate):
            video_path = candidate
            break

    sessions.append(dict(
        csv_path   = csv_path,
        video_path = video_path,
        session_id = session_id,
    ))

print(f"Found {len(sessions)} Lightning Pose session(s):")
for s in sessions:
    tag = "" if s["video_path"] else "  [no video]"
    print(f"  {s['session_id']}{tag}")


# Load All Sessions
Lightning Pose CSVs share the same `scorer` / `bodyparts` / `coords` header
layout as DLC exports, so they load the same way — the `scorer` row here
reads `heatmap_tracker` rather than a `DLC_...` string, and is kept per
session so the corrected output can be re-wrapped into that same format later.

In [ ]:
# Load every session's Lightning Pose predictions and maze background image.
# Results are stored back into the sessions list.
for s in sessions:
    print(f"Loading {s['session_id']} ...", end=" ")

    # Maze background (first frame of video), if a video is available
    if s["video_path"]:
        cap = cv2.VideoCapture(s["video_path"])
        ret, frame = cap.read()
        cap.release()
        s["maze_image"] = frame if ret else None
    else:
        s["maze_image"] = None

    # Lightning Pose data (same scorer/bodyparts/coords layout as DLC)
    raw = pd.read_csv(s["csv_path"], header=[0, 1, 2], index_col=0)
    scorer_name = raw.columns.get_level_values(0)[0]
    dlc = raw[scorer_name].apply(pd.to_numeric, errors="coerce")
    s["scorer_name"] = scorer_name
    s["dlc"] = dlc

    s["headstage_x"]   = dlc["headstage"]["x"].values
    s["headstage_y"]   = dlc["headstage"]["y"].values
    s["headstage_lik"] = dlc["headstage"]["likelihood"].values

    n  = len(s["headstage_x"])
    nl = int((s["headstage_lik"] < LIKELIHOOD_THR).sum())
    print(f"{n} frames  |  {nl} low-lik ({100*nl/n:.1f}%)")


# Headstage Likelihood Distributions (Lightning Pose)
One subplot per session, showing the full likelihood histogram and the
correction threshold.

In [ ]:
n_sess = len(sessions)
ncols  = min(3, n_sess)
nrows  = (n_sess + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False)
fig.suptitle("Headstage likelihood distributions (Lightning Pose)", fontsize=14, fontweight="bold")

for idx, s in enumerate(sessions):
    ax  = axes[idx // ncols][idx % ncols]
    lik = s["headstage_lik"]
    n_low = int((lik < LIKELIHOOD_THR).sum())

    ax.hist(lik, bins=100, color="darkorange", edgecolor="none", alpha=0.8)
    ax.axvline(LIKELIHOOD_THR, color="red", lw=1.5, ls="--",
               label=f"Threshold {LIKELIHOOD_THR}")
    ax.set_title(s["session_id"].split("_stitched")[0], fontsize=9)
    ax.set_xlabel("Likelihood")
    ax.set_ylabel("Frames")
    ax.legend(fontsize=7)
    ax.text(0.02, 0.97, f"{n_low} frames below threshold ({100*n_low/len(lik):.1f}%)",
            transform=ax.transAxes, va="top", fontsize=7, color="red")

# Hide unused subplots
for idx in range(n_sess, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

plt.tight_layout()
os.makedirs(OUTPUT_DIR, exist_ok=True)
plt.savefig(os.path.join(OUTPUT_DIR, "likelihood_distributions.png"),
            dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Mean likelihood distribution across all sessions
# Stack all sessions into a single histogram, then overlay the per-session mean

fig, ax = plt.subplots(figsize=(8, 5))

bins = np.linspace(0, 1, 101)
bin_centers = (bins[:-1] + bins[1:]) / 2

all_hists = []
for s in sessions:
    counts, _ = np.histogram(s["headstage_lik"], bins=bins, density=True)
    all_hists.append(counts)
    ax.plot(bin_centers, counts, color="darkorange", lw=0.8, alpha=0.3)

mean_hist = np.mean(all_hists, axis=0)
std_hist  = np.std(all_hists, axis=0)

ax.plot(bin_centers, mean_hist, color="darkorange", lw=2.5, label="Mean across sessions")
ax.fill_between(bin_centers,
                np.maximum(0, mean_hist - std_hist),
                mean_hist + std_hist,
                color="darkorange", alpha=0.25, label="±1 SD")
ax.axvline(LIKELIHOOD_THR, color="red", lw=1.5, ls="--",
           label=f"Threshold {LIKELIHOOD_THR}")

ax.set_xlabel("Likelihood", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title(f"Mean headstage likelihood distribution — Lightning Pose ({len(sessions)} sessions)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim(0, 1)

plt.tight_layout()
os.makedirs(OUTPUT_DIR, exist_ok=True)
plt.savefig(os.path.join(OUTPUT_DIR, "likelihood_mean_distribution.png"),
            dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Standalone DLC-vs-DLC comparison — unrelated to the Lightning Pose correction
# pipeline above, kept self-contained via DLC_LEGACY_FOLDER.
SHUFFLE_A = dict(num=6, color="steelblue",  label="shuffle6")
SHUFFLE_B = dict(num=7, color="darkorange", label="shuffle7")

bins        = np.linspace(0, 1, 101)
bin_centers = (bins[:-1] + bins[1:]) / 2

# ── Auto-discover sessions that have BOTH shuffles ─────────────────────────────
re_shuf = re.compile(r"shuffle(\d+)_snapshot_\d+\.h5$", re.IGNORECASE)

shuf_map = {}   # prefix -> {shuffle_num: (h5_path, table_name)}
for h5 in sorted(glob.glob(os.path.join(DLC_LEGACY_FOLDER, "*DLC*.h5"))):
    fname = os.path.basename(h5)
    if fname.endswith("_filtered.h5") or fname.endswith("_corrected.h5"):
        continue
    m = re_shuf.search(fname)
    if not m:
        continue
    snum = int(m.group(1))
    if snum not in (SHUFFLE_A["num"], SHUFFLE_B["num"]):
        continue
    m2 = re.search(r"(DLC_.+)$", fname[:-3])
    if not m2:
        continue
    table_name = m2.group(1)
    prefix = fname[: fname.index("DLC_")]
    shuf_map.setdefault(prefix, {})[snum] = (h5, table_name)

paired = {
    p: v for p, v in shuf_map.items()
    if SHUFFLE_A["num"] in v and SHUFFLE_B["num"] in v
}
print(f"Sessions with both shuffle{SHUFFLE_A['num']} and shuffle{SHUFFLE_B['num']}: {len(paired)}")
for p in sorted(paired):
    print(f"  {p.rstrip('_')}")

# ── Load likelihood data ───────────────────────────────────────────────────────
session_data = []
for prefix in sorted(paired):
    session_label = prefix.rstrip("_").split("_stitched")[0]
    row = {"session": session_label}
    for shuf_info in (SHUFFLE_A, SHUFFLE_B):
        snum = shuf_info["num"]
        h5_path, table_name = paired[prefix][snum]
        dlc = pd.read_hdf(h5_path)[table_name].apply(pd.to_numeric, errors="coerce")
        lik = dlc["headstage"]["likelihood"].values
        counts, _ = np.histogram(lik, bins=bins, density=True)
        n_low = int((lik < LIKELIHOOD_THR).sum())
        row[f"sh{snum}_counts"]  = counts
        row[f"sh{snum}_mean"]    = float(np.nanmean(lik))
        row[f"sh{snum}_pct_low"] = 100 * n_low / len(lik)
    session_data.append(row)

# ── Per-session subplot grid ───────────────────────────────────────────────────
n_sess = len(session_data)

if n_sess == 0:
    print("No sessions with both shuffles present — skipping comparison plot.")
else:
    ncols  = min(3, n_sess)
    nrows  = (n_sess + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False)
    fig.suptitle(
        f"Per-session: shuffle{SHUFFLE_A['num']} vs shuffle{SHUFFLE_B['num']}",
        fontsize=14, fontweight="bold",
    )

    for idx, row in enumerate(session_data):
        ax = axes[idx // ncols][idx % ncols]
        for shuf_info in (SHUFFLE_A, SHUFFLE_B):
            snum = shuf_info["num"]
            ax.plot(
                bin_centers, row[f"sh{snum}_counts"],
                color=shuf_info["color"], lw=1.8,
                label=f"sh{snum}  μ={row[f'sh{snum}_mean']:.3f}  "
                      f"{row[f'sh{snum}_pct_low']:.1f}%<thr",
            )
            ax.axvline(row[f"sh{snum}_mean"], color=shuf_info["color"], lw=1, ls=":")
        ax.axvline(LIKELIHOOD_THR, color="red", lw=1.2, ls="--")
        ax.set_title(row["session"], fontsize=8)
        ax.set_xlabel("Likelihood")
        ax.set_ylabel("Density")
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)
        ax.set_xlim(0, 1)

    for idx in range(n_sess, nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    plt.tight_layout()
    plt.savefig(
        os.path.join(DLC_LEGACY_FOLDER,
                     f"shuffle{SHUFFLE_A['num']}_vs_shuffle{SHUFFLE_B['num']}_per_session.png"),
        dpi=120, bbox_inches="tight",
    )
    plt.show()


In [ ]:
# ── Aggregated: mean ± SD across all paired sessions ──────────────────────────
sa, sb = SHUFFLE_A["num"], SHUFFLE_B["num"]

if not session_data:
    print("No sessions with both shuffles present — skipping aggregated plot and summary table.")
else:
    fig, ax = plt.subplots(figsize=(9, 5))

    for shuf_info in (SHUFFLE_A, SHUFFLE_B):
        snum   = shuf_info["num"]
        stack  = np.array([row[f"sh{snum}_counts"] for row in session_data])
        mean_h = stack.mean(axis=0)
        std_h  = stack.std(axis=0)
        mean_pct_low = np.mean([row[f"sh{snum}_pct_low"] for row in session_data])
        for c in stack:
            ax.plot(bin_centers, c, color=shuf_info["color"], lw=0.7, alpha=0.2)
        ax.plot(
            bin_centers, mean_h,
            color=shuf_info["color"], lw=2.5,
            label=f"shuffle{snum}  mean ± SD  (avg {mean_pct_low:.1f}% < thr)",
        )
        ax.fill_between(
            bin_centers,
            np.maximum(0, mean_h - std_h),
            mean_h + std_h,
            color=shuf_info["color"], alpha=0.2,
        )

    ax.axvline(LIKELIHOOD_THR, color="red", lw=1.5, ls="--",
               label=f"Threshold {LIKELIHOOD_THR}")
    ax.set_xlabel("Likelihood", fontsize=12)
    ax.set_ylabel("Density", fontsize=12)
    ax.set_title(
        f"Aggregated ({len(session_data)} sessions): "
        f"shuffle{sa} vs shuffle{sb}",
        fontsize=13, fontweight="bold",
    )
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    ax.set_xlim(0, 1)
    plt.tight_layout()
    plt.savefig(
        os.path.join(DLC_LEGACY_FOLDER,
                     f"shuffle{sa}_vs_shuffle{sb}_aggregated.png"),
        dpi=120, bbox_inches="tight",
    )
    plt.show()

    # ── Summary table ──────────────────────────────────────────────────────────
    print(f"\n{'Session':<45} {f'sh{sa} mean':>9} {f'sh{sa} %<thr':>11} "
          f"{f'sh{sb} mean':>9} {f'sh{sb} %<thr':>11}")
    print("-" * 90)
    for row in session_data:
        print(f"{row['session']:<45} "
              f"{row[f'sh{sa}_mean']:>9.3f} {row[f'sh{sa}_pct_low']:>10.1f}%  "
              f"{row[f'sh{sb}_mean']:>9.3f} {row[f'sh{sb}_pct_low']:>10.1f}%")


# Shuffle 7 vs Shuffle 8 Comparison
Same likelihood-distribution comparison as above, but for shuffle7 vs the new
shuffle8 model. Unlike shuffle6/7 (which live in a single folder), the two
models here live in different folders and shuffle7 has multiple snapshot
exports floating around, so discovery is more general:

- **shuffle7** is scanned from `DLC_LEGACY_FOLDER` (`NPC4_new`) and
  `C:\Maze\Video Data\NPC4\` (older shuffle7 exports for sessions not in
  `NPC4_new`).
- **shuffle8** is scanned from
  `C:\Users\Almog\Desktop\Neuropixel_2Cams_Gamma-AnA_July-2025-07-22\videos\`.
- When a session has more than one snapshot for the same shuffle (e.g.
  shuffle7 snapshot_140 and snapshot_500), the **highest snapshot number** is
  used.
- Only sessions present for **both** shuffles are compared; a shuffle8 session
  with no shuffle7 counterpart (or vice versa) is skipped.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
SHUFFLE7_FOLDERS = [DLC_LEGACY_FOLDER, r"C:\\Maze\\Video Data\\NPC4\\"]
SHUFFLE8_FOLDERS = [r"C:\\Users\\Almog\\Desktop\\Neuropixel_2Cams_Gamma-AnA_July-2025-07-22\\videos\\"]

SHUFFLE_A = dict(num=7, color="steelblue",  label="shuffle7", folders=SHUFFLE7_FOLDERS)
SHUFFLE_B = dict(num=8, color="darkorange", label="shuffle8", folders=SHUFFLE8_FOLDERS)

bins        = np.linspace(0, 1, 101)
bin_centers = (bins[:-1] + bins[1:]) / 2


def find_shuffle_files(folders, shuffle_num):
    """Scan multiple folders for one shuffle's h5 files, excluding
    filtered/corrected/updated variants. Returns {session_prefix: (h5_path, table_name)},
    keeping the highest snapshot number per session when duplicates exist."""
    pattern = re.compile(rf"shuffle{shuffle_num}_snapshot_(\d+)\.h5$", re.IGNORECASE)
    best_snap = {}
    found = {}
    for folder in folders:
        for h5 in sorted(glob.glob(os.path.join(folder, "*DLC*.h5"))):
            fname = os.path.basename(h5)
            if fname.endswith(("_filtered.h5", "_corrected.h5", "_updated.h5")):
                continue
            m = pattern.search(fname)
            if not m:
                continue
            snap = int(m.group(1))
            m2 = re.search(r"(DLC_.+)$", fname[:-3])
            if not m2:
                continue
            table_name = m2.group(1)
            prefix = fname[: fname.index("DLC_")].rstrip("_")
            if prefix not in best_snap or snap > best_snap[prefix]:
                best_snap[prefix] = snap
                found[prefix] = (h5, table_name)
    return found


# ── Auto-discover sessions that have BOTH shuffles ─────────────────────────────
files_a = find_shuffle_files(SHUFFLE_A["folders"], SHUFFLE_A["num"])
files_b = find_shuffle_files(SHUFFLE_B["folders"], SHUFFLE_B["num"])

paired = {p: {SHUFFLE_A["num"]: files_a[p], SHUFFLE_B["num"]: files_b[p]}
          for p in sorted(set(files_a) & set(files_b))}

print(f"Sessions with both shuffle{SHUFFLE_A['num']} and shuffle{SHUFFLE_B['num']}: {len(paired)}")
for p in sorted(paired):
    print(f"  {p}")

skipped_b_only = sorted(set(files_b) - set(files_a))
if skipped_b_only:
    print(f"\nshuffle{SHUFFLE_B['num']} sessions with no shuffle{SHUFFLE_A['num']} match (skipped):")
    for p in skipped_b_only:
        print(f"  {p}")

# ── Load likelihood data ───────────────────────────────────────────────────────
session_data = []
for prefix in sorted(paired):
    session_label = prefix.split("_stitched")[0]
    row = {"session": session_label}
    for shuf_info in (SHUFFLE_A, SHUFFLE_B):
        snum = shuf_info["num"]
        h5_path, table_name = paired[prefix][snum]
        dlc = pd.read_hdf(h5_path)[table_name].apply(pd.to_numeric, errors="coerce")
        lik = dlc["headstage"]["likelihood"].values
        counts, _ = np.histogram(lik, bins=bins, density=True)
        n_low = int((lik < LIKELIHOOD_THR).sum())
        row[f"sh{snum}_counts"]  = counts
        row[f"sh{snum}_mean"]    = float(np.nanmean(lik))
        row[f"sh{snum}_pct_low"] = 100 * n_low / len(lik)
    session_data.append(row)

# ── Per-session subplot grid ───────────────────────────────────────────────────
n_sess = len(session_data)

if n_sess == 0:
    print("No sessions with both shuffles present — skipping comparison plot.")
else:
    ncols  = min(3, n_sess)
    nrows  = (n_sess + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False)
    fig.suptitle(
        f"Per-session: shuffle{SHUFFLE_A['num']} vs shuffle{SHUFFLE_B['num']}",
        fontsize=14, fontweight="bold",
    )

    for idx, row in enumerate(session_data):
        ax = axes[idx // ncols][idx % ncols]
        for shuf_info in (SHUFFLE_A, SHUFFLE_B):
            snum = shuf_info["num"]
            ax.plot(
                bin_centers, row[f"sh{snum}_counts"],
                color=shuf_info["color"], lw=1.8,
                label=f"sh{snum}  μ={row[f'sh{snum}_mean']:.3f}  "
                      f"{row[f'sh{snum}_pct_low']:.1f}%<thr",
            )
            ax.axvline(row[f"sh{snum}_mean"], color=shuf_info["color"], lw=1, ls=":")
        ax.axvline(LIKELIHOOD_THR, color="red", lw=1.2, ls="--")
        ax.set_title(row["session"], fontsize=8)
        ax.set_xlabel("Likelihood")
        ax.set_ylabel("Density")
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)
        ax.set_xlim(0, 1)

    for idx in range(n_sess, nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    plt.tight_layout()
    plt.savefig(
        os.path.join(DLC_LEGACY_FOLDER,
                     f"shuffle{SHUFFLE_A['num']}_vs_shuffle{SHUFFLE_B['num']}_per_session.png"),
        dpi=120, bbox_inches="tight",
    )
    plt.show()


In [ ]:
# ── Aggregated: mean ± SD across all paired sessions ──────────────────────────
sa, sb = SHUFFLE_A["num"], SHUFFLE_B["num"]

if not session_data:
    print("No sessions with both shuffles present — skipping aggregated plot and summary table.")
else:
    fig, ax = plt.subplots(figsize=(9, 5))

    for shuf_info in (SHUFFLE_A, SHUFFLE_B):
        snum   = shuf_info["num"]
        stack  = np.array([row[f"sh{snum}_counts"] for row in session_data])
        mean_h = stack.mean(axis=0)
        std_h  = stack.std(axis=0)
        mean_pct_low = np.mean([row[f"sh{snum}_pct_low"] for row in session_data])
        for c in stack:
            ax.plot(bin_centers, c, color=shuf_info["color"], lw=0.7, alpha=0.2)
        ax.plot(
            bin_centers, mean_h,
            color=shuf_info["color"], lw=2.5,
            label=f"shuffle{snum}  mean ± SD  (avg {mean_pct_low:.1f}% < thr)",
        )
        ax.fill_between(
            bin_centers,
            np.maximum(0, mean_h - std_h),
            mean_h + std_h,
            color=shuf_info["color"], alpha=0.2,
        )

    ax.axvline(LIKELIHOOD_THR, color="red", lw=1.5, ls="--",
               label=f"Threshold {LIKELIHOOD_THR}")
    ax.set_xlabel("Likelihood", fontsize=12)
    ax.set_ylabel("Density", fontsize=12)
    ax.set_title(
        f"Aggregated ({len(session_data)} sessions): "
        f"shuffle{sa} vs shuffle{sb}",
        fontsize=13, fontweight="bold",
    )
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    ax.set_xlim(0, 1)
    plt.tight_layout()
    plt.savefig(
        os.path.join(DLC_LEGACY_FOLDER,
                     f"shuffle{sa}_vs_shuffle{sb}_aggregated.png"),
        dpi=120, bbox_inches="tight",
    )
    plt.show()

    # ── Summary table ──────────────────────────────────────────────────────────
    print(f"\n{'Session':<45} {f'sh{sa} mean':>9} {f'sh{sa} %<thr':>11} "
          f"{f'sh{sb} mean':>9} {f'sh{sb} %<thr':>11}")
    print("-" * 90)
    for row in session_data:
        print(f"{row['session']:<45} "
              f"{row[f'sh{sa}_mean']:>9.3f} {row[f'sh{sa}_pct_low']:>10.1f}%  "
              f"{row[f'sh{sb}_mean']:>9.3f} {row[f'sh{sb}_pct_low']:>10.1f}%")


# Piecewise Linear Correction
Ported from `jumps_skeleton.ipynb`'s `estimate_low_likelihood_positions_piecewise`.
For each low-likelihood frame the two nearest high-likelihood anchor frames
(before and after) are found within the 10-second chunk context and the
position is linearly interpolated. The context extends one chunk into each
neighbour so edge frames are handled correctly.

`jumps_skeleton.ipynb` partitions frames by behavioral trial (from a per-session
`trial_info.mat`); no such file exists for any NPC4 session, so here the same
10-second chunks used elsewhere in this notebook stand in for trials. One
behavior is carried over as-is: a chunk is skipped (left uncorrected) if the
mouse's x/y position range within it is below `MIN_MOVEMENT_PX` in both axes —
i.e. near-stationary periods aren't interpolated.

In [11]:
def correct_low_likelihood_piecewise(
        x, y, likelihood, chunk_starts, chunk_ends,
        likelihood_threshold=0.85, min_movement=10):
    """
    Correct low-likelihood DLC frames via piecewise linear interpolation.
    Ported from jumps_skeleton.ipynb's estimate_low_likelihood_positions_piecewise:
    chunks with less than `min_movement` px of x/y range (i.e. the mouse barely
    moved) are skipped and left uncorrected, same as that notebook's trial-level
    min_movement filter.
    Returns corrected x/y arrays, a boolean mask of corrected frames,
    a list of skipped chunk indices, and the total count of corrected frames.
    """
    x_corr       = x.copy()
    y_corr       = y.copy()
    low_lik_mask = likelihood < likelihood_threshold
    chunks_skipped = []
    n_corrected    = 0
    n_chunks       = len(chunk_starts)

    for ci in range(n_chunks):
        cs = int(chunk_starts[ci])
        ce = int(chunk_ends[ci])
        if cs >= ce:
            chunks_skipped.append(ci)
            continue

        # Skip near-stationary chunks (mouse isn't moving enough to need correction)
        chunk_x_range = np.max(x[cs:ce]) - np.min(x[cs:ce])
        chunk_y_range = np.max(y[cs:ce]) - np.min(y[cs:ce])
        if chunk_x_range < min_movement and chunk_y_range < min_movement:
            chunks_skipped.append(ci)
            continue

        # Extend context one chunk on each side
        ctx_start = int(chunk_starts[ci - 1]) if ci > 0 else cs
        ctx_end   = int(chunk_ends[ci + 1])   if ci < n_chunks - 1 else ce

        ctx_lik    = likelihood[ctx_start:ctx_end]
        ctx_frames = np.arange(ctx_start, ctx_end)
        hi_idx     = np.where(ctx_lik >= likelihood_threshold)[0]

        if len(hi_idx) < 2:
            chunks_skipped.append(ci)
            continue

        for gf in range(cs, ce):
            if not low_lik_mask[gf]:
                continue
            ci_ctx = gf - ctx_start
            before = hi_idx[hi_idx < ci_ctx]
            after  = hi_idx[hi_idx > ci_ctx]
            if len(before) == 0 or len(after) == 0:
                continue
            fb  = int(ctx_frames[before[-1]])
            fa  = int(ctx_frames[after[0]])
            gap = fa - fb
            if gap == 0:
                continue
            w           = (gf - fb) / gap
            x_corr[gf] = x[fb] + w * (x[fa] - x[fb])
            y_corr[gf] = y[fb] + w * (y[fa] - y[fb])
            n_corrected += 1

    return x_corr, y_corr, low_lik_mask, chunks_skipped, n_corrected


# Correct, Plot, and Save — All Sessions
For each session: runs the correction, saves chunk plots for the most-affected
chunks, then writes `<session>_<scorer>_corrected.h5` and `.csv` into
`OUTPUT_DIR`. Both are re-wrapped with the original `scorer` level
(`heatmap_tracker`) on top of `bodyparts`/`coords`, matching the DLC
`scorer`/`bodyparts`/`coords` layout — the same format `DLC_H5_Update.ipynb`
produces — so the output is a drop-in NaviGraph input.

In [ ]:
chunk_size = FPS * CHUNK_SEC
os.makedirs(OUTPUT_DIR, exist_ok=True)

for s in sessions:
    sid  = s["session_id"]
    x    = s["headstage_x"]
    y    = s["headstage_y"]
    lik  = s["headstage_lik"]
    n_frames = len(x)

    print(f"\n=== {sid} ===")

    # Build chunk boundaries
    chunk_starts = np.arange(0, n_frames, chunk_size)
    chunk_ends   = np.minimum(chunk_starts + chunk_size, n_frames)

    # Correct
    x_corr, y_corr, low_lik_mask, chunks_skipped, n_corrected = \
        correct_low_likelihood_piecewise(
            x, y, lik, chunk_starts, chunk_ends,
            likelihood_threshold=LIKELIHOOD_THR,
            min_movement=MIN_MOVEMENT_PX,
        )
    print(f"  Frames corrected : {n_corrected}")
    print(f"  Chunks skipped   : {len(chunks_skipped)}")

    # ── Plots ──────────────────────────────────────────────────────────
    plots_dir = os.path.join(OUTPUT_DIR, f"correction_plots_{sid.split('_stitched')[0]}")
    os.makedirs(plots_dir, exist_ok=True)

    low_lik_counts = [
        (ci, int(low_lik_mask[int(chunk_starts[ci]):int(chunk_ends[ci])].sum()))
        for ci in range(len(chunk_starts))
    ]
    low_lik_counts.sort(key=lambda t: t[1], reverse=True)
    chunks_to_plot = [ci for ci, n in low_lik_counts[:N_PLOT_CHUNKS] if n > 0]
    print(f"  Saving {len(chunks_to_plot)} chunk plots -> {plots_dir}")

    maze_img = s.get("maze_image")
    for ci in chunks_to_plot:
        cs, ce  = int(chunk_starts[ci]), int(chunk_ends[ci])
        time_s  = np.arange(cs, ce) / FPS
        raw_x   = x[cs:ce];   raw_y  = y[cs:ce]
        corr_x  = x_corr[cs:ce]; corr_y = y_corr[cs:ce]
        lk      = lik[cs:ce]
        low     = low_lik_mask[cs:ce]
        n_low   = int(low.sum())

        fig, axes = plt.subplots(2, 2, figsize=(16, 10))
        fig.suptitle(
            f"{sid.split('_stitched')[0]}  |  Chunk {ci}  "
            f"t = {cs/FPS:.1f}–{ce/FPS:.1f} s  |  "
            f"{n_low} frames corrected",
            fontsize=12, fontweight="bold",
        )

        ax = axes[0, 0]
        ax.plot(time_s, raw_x,  color="steelblue", lw=1, alpha=0.7, label="Original")
        ax.plot(time_s, corr_x, color="coral",     lw=1, alpha=0.9, label="Corrected")
        ax.scatter(time_s[low], raw_x[low], color="red", s=20, zorder=5, label="Low lik")
        ax.set_xlabel("Time (s)"); ax.set_ylabel("X (px)")
        ax.set_title("X coordinate"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

        ax = axes[0, 1]
        ax.plot(time_s, raw_y,  color="steelblue", lw=1, alpha=0.7, label="Original")
        ax.plot(time_s, corr_y, color="coral",     lw=1, alpha=0.9, label="Corrected")
        ax.scatter(time_s[low], raw_y[low], color="red", s=20, zorder=5, label="Low lik")
        ax.set_xlabel("Time (s)"); ax.set_ylabel("Y (px)")
        ax.set_title("Y coordinate"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

        ax = axes[1, 0]
        ax.plot(time_s, lk, color="steelblue", lw=1)
        ax.axhline(LIKELIHOOD_THR, color="red", lw=1, ls="--",
                   label=f"Threshold {LIKELIHOOD_THR}")
        ax.fill_between(time_s, 0, 1, where=low,
                        color="red", alpha=0.15, label="Corrected region")
        ax.set_xlabel("Time (s)"); ax.set_ylabel("Likelihood")
        ax.set_title("Tracking likelihood")
        ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_ylim(0, 1.05)

        ax = axes[1, 1]
        if maze_img is not None:
            ax.imshow(maze_img)
        ax.plot(raw_x,  raw_y,  color="steelblue", lw=0.8, alpha=0.5, label="Original")
        ax.plot(corr_x, corr_y, color="coral",     lw=0.8, alpha=0.8, label="Corrected")
        if low.any():
            ax.scatter(raw_x[low], raw_y[low], color="red", s=15, zorder=5,
                       label="Low lik (raw)")
        ax.set_title("Trajectory on maze"); ax.legend(fontsize=8); ax.axis("off")

        plt.tight_layout()
        plt.savefig(os.path.join(plots_dir, f"chunk_{ci:04d}.png"),
                    dpi=100, bbox_inches="tight")
        plt.close()

    # ── Save (DLC-format h5 + csv, for NaviGraph) ─────────────────────────
    dlc = s["dlc"]
    dlc[("headstage", "x")] = x_corr
    dlc[("headstage", "y")] = y_corr

    scorer_name = s["scorer_name"]
    out_df = pd.concat({scorer_name: dlc}, axis=1)
    out_df.columns.names = ["scorer", "bodyparts", "coords"]

    base = os.path.join(OUTPUT_DIR, f"{sid}_{scorer_name}_corrected")
    out_df.to_hdf(base + ".h5", key=scorer_name, mode="w")
    print(f"  Saved: {base}.h5")
    out_df.to_csv(base + ".csv")
    print(f"  Saved: {base}.csv")

print("\nAll sessions done.")
